# 9. Summary of all results

Collects every 5-fold result: `runs/<model>_fold<k>/metrics.json` (models A and B) and `runs/wdsvm/*.json`
(models C, D, E). Test writers are never seen in training; thresholds come from the validation writers.
Writes `runs/summary.csv`.

In [1]:
%run ./common.ipynb

In [2]:
import glob
import json

KEYS = ["auc", "eer", "acc", "far", "frr"]
rows = []

per_fold = {}
for f in sorted(glob.glob(os.path.join(RUNS, "*_fold*", "metrics.json"))):
    r = json.load(open(f))
    per_fold.setdefault(r["mode"], []).append(r["test"])
for mode, folds in per_fold.items():
    if len(folds) != 5:
        continue
    df = pd.DataFrame(folds)
    label = {"binary": "A. single-image classifier", "siamese": "B. Siamese (5 refs)"}.get(mode, mode)
    rows.append({"model": label, **{k: f"{df[k].mean():.3f} +/- {df[k].std(ddof=0):.3f}" for k in KEYS},
                 "per-writer eer": "-"})

for f in sorted(glob.glob(os.path.join(RUNS, "wdsvm", "*.json"))):
    r = json.load(open(f))
    rows.append({"model": f"WD-SVM {r['features']} feats, {r['mode']} input, {r['n_ref']} refs",
                 **{k: f"{r[k + '_mean']:.3f} +/- {r[k + '_std']:.3f}" for k in KEYS},
                 "per-writer eer": f"{r['mean_writer_eer_mean']:.3f} +/- {r['mean_writer_eer_std']:.3f}"})

summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(RUNS, "summary.csv"), index=False)
summary

,model,auc,eer,acc,far,frr,per-writer eer
0,A. single-image classifier,0.751 +/- 0.046,0.320 +/- 0.034,0.668 +/- 0.045,0.262 +/- 0.155,0.402 +/- 0.197,-
1,B. Siamese (5 refs),0.702 +/- 0.057,0.366 +/- 0.052,0.626 +/- 0.042,0.359 +/- 0.135,0.393 +/- 0.159,-
2,"WD-SVM binary feats, binary input, 5 refs",0.860 +/- 0.048,0.216 +/- 0.052,0.756 +/- 0.074,0.305 +/- 0.161,0.166 +/- 0.069,0.167 +/- 0.029
3,"WD-SVM imagenet feats, binary input, 5 refs",0.903 +/- 0.024,0.176 +/- 0.027,0.821 +/- 0.034,0.201 +/- 0.063,0.151 +/- 0.026,0.132 +/- 0.018
4,"WD-SVM imagenet feats, skel input, 5 refs",0.879 +/- 0.021,0.196 +/- 0.028,0.798 +/- 0.030,0.177 +/- 0.025,0.233 +/- 0.045,0.153 +/- 0.016
5,"WD-SVM writeridF feats, binary input, 5 refs",0.911 +/- 0.024,0.171 +/- 0.034,0.807 +/- 0.043,0.241 +/- 0.097,0.132 +/- 0.034,0.111 +/- 0.020
6,"WD-SVM writerid feats, binary input, 1 refs",0.870 +/- 0.043,0.213 +/- 0.045,0.776 +/- 0.046,0.299 +/- 0.144,0.145 +/- 0.080,0.140 +/- 0.043
7,"WD-SVM writerid feats, binary input, 10 refs",0.944 +/- 0.016,0.130 +/- 0.039,0.860 +/- 0.032,0.172 +/- 0.052,0.084 +/- 0.020,0.100 +/- 0.029
8,"WD-SVM writerid feats, binary input, 3 refs",0.922 +/- 0.013,0.155 +/- 0.019,0.834 +/- 0.044,0.231 +/- 0.111,0.091 +/- 0.050,0.109 +/- 0.017
9,"WD-SVM writerid feats, binary input, 5 refs",0.931 +/- 0.018,0.138 +/- 0.029,0.844 +/- 0.045,0.208 +/- 0.097,0.091 +/- 0.026,0.099 +/- 0.016
